# 01 - Análise exploratória do N1

**Etapa:** primeiro bimestre (N1).

**Integrantes:** Alan Ribeiro do Carmo (RA 10428496), Jean Pazzini Domingues (RA 10428555) e Wendell Rodrigues da Costa (RA 10420319).

**Docente/orientador:** Leandro Zerbinatti (leandro.zerbinatti@mackenzie.br).

**Síntese:** carregamento, inspeção, qualidade, estatísticas descritivas, visualizações e relações exploratórias.

**Histórico:** 2026-09-15 | GitHub Copilot | Adequação da EDA ao escopo N1.

Este notebook não treina modelos preditivos nem calcula métricas de Machine Learning. A classificação de satisfação é criada apenas para análise exploratória, conforme a regra oficial baseada na nota.

In [1]:
from pathlib import Path
import sys
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.data_loader import load_dataset, validate_and_create_target
from src.visualization import plot_exploration

DATA = ROOT / 'data' / 'raw' / 'dataset_satisfacao_atendimento.csv'
OUTPUT = ROOT / 'results' / 'figures'

data = load_dataset(DATA)
prepared, quality = validate_and_create_target(data)

print('Dimensão:', prepared.shape)
print('\nTipos das colunas:')
display(prepared.dtypes.rename('tipo').to_frame())
print('\nValores ausentes por coluna:')
display(prepared.isna().sum().rename('ausentes').to_frame())
print('\nLinhas duplicadas:', int(prepared.duplicated().sum()))
print('IDs duplicados:', int(prepared['ID_Resposta'].duplicated().sum()))
print('\nDistribuição da satisfação derivada:')
display(prepared['Satisfacao'].map({0: 'Nao-Satisfeito', 1: 'Satisfeito'}).value_counts().to_frame('quantidade'))
print('\nDistribuição da nota de atendimento:')
display(prepared['Nota_para_o_atendimento_1_a_5'].value_counts().sort_index().to_frame('quantidade'))
print('\nEstatísticas descritivas:')
display(prepared.describe(include='all').T)
print('\nRelatório de consistência do target:')
display(pd.Series(quality, name='valor').to_frame())

plot_exploration(prepared, OUTPUT)

satisfied = prepared.groupby('Satisfacao')['Tempo_de_espera_minutos'].mean()
resolved = pd.crosstab(prepared['Problema_foi_resolvido'], prepared['Satisfacao'], normalize='index')
print('\nInterpretação descritiva:')
print(f"- O tempo médio de espera foi {satisfied.get(1, float('nan')):.2f} min para Satisfeito e {satisfied.get(0, float('nan')):.2f} min para Nao-Satisfeito.")
print(f"- Entre respostas com problema resolvido, a proporção satisfeita foi {resolved.loc['Sim', 1] * 100:.1f}%.")
print('- Os gráficos por faixa etária, canal, resolução, novo contato, cordialidade, facilidade e recompra descrevem associações observadas; não representam causalidade.')
print(f'Gráficos salvos em: {OUTPUT}')

Dimensão: (207, 11)

Tipos das colunas:


,tipo
ID_Resposta,int64
Faixa_etaria,str
Canal_de_atendimento,str
Tempo_de_espera_minutos,int64
Problema_foi_resolvido,str
Precisou_contato_novamente,str
Atendimento_foi_cordial,str
Facilidade_para_resolver_1_a_5,int64
Nota_para_o_atendimento_1_a_5,int64
Voltaria_a_comprar,str



Valores ausentes por coluna:


,ausentes
ID_Resposta,0
Faixa_etaria,0
Canal_de_atendimento,0
Tempo_de_espera_minutos,0
Problema_foi_resolvido,0
Precisou_contato_novamente,0
Atendimento_foi_cordial,0
Facilidade_para_resolver_1_a_5,0
Nota_para_o_atendimento_1_a_5,0
Voltaria_a_comprar,0



Linhas duplicadas: 0
IDs duplicados: 0

Distribuição da satisfação derivada:


...

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
ID_Resposta,207.0,NaN,NaN,NaN,104.0,59.899917,1.0,52.5,104.0,155.5,207.0
Faixa_etaria,207,4,26-40,75,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Canal_de_atendimento,207,5,WhatsApp,80,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Tempo_de_espera_minutos,207.0,NaN,NaN,NaN,9.797101,8.577878,1.0,4.0,7.0,13.0,46.0
Problema_foi_resolvido,207,2,Sim,172,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Precisou_contato_novamente,207,2,Nao,171,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Atendimento_foi_cordial,207,2,Sim,163,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Facilidade_para_resolver_1_a_5,207.0,NaN,NaN,NaN,3.038647,1.127095,1.0,2.0,3.0,4.0,5.0
Nota_para_o_atendimento_1_a_5,207.0,NaN,NaN,NaN,4.193237,0.919768,1.0,4.0,4.0,5.0,5.0
Voltaria_a_comprar,207,2,Sim,157,NaN,NaN,NaN,NaN,NaN,NaN,NaN



Relatório de consistência do target:


,valor
shape,"[207, 11]"
missing_values,"{'ID_Resposta': 0, 'Faixa_etaria': 0, 'Canal_d..."
duplicate_rows,0
duplicate_ids,0
target_inconsistencies,31
target_original_values,"[Nao-Satisfeito, Satisfeito]"



Interpretação descritiva:
- O tempo médio de espera foi 9.80 min para Satisfeito e 9.80 min para Nao-Satisfeito.
- Entre respostas com problema resolvido, a proporção satisfeita foi 87.8%.
- Os gráficos por faixa etária, canal, resolução, novo contato, cordialidade, facilidade e recompra descrevem associações observadas; não representam causalidade.
Gráficos salvos em: /workspaces/previsao-satisfacao-clientes-ml/results/figures


<Figure size 800x400 with 0 Axes>